In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
MODEL      = 'MIT-B5'       # MIT-B5 | MIT-B2 | MIT-B3
IMG_SIZE   = (768, 768)
BATCH_SIZE = 8
EPOCHS     = 100
LR         = 3e-4
FOCAL_LOSS = True
PATIENCE   = 20

RESUME = None

TRAIN_SEQS = ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8']
VAL_SEQS   = ['seq6']

_ZOO = {'MIT-B5': 'mit_b5', 'MIT-B2': 'mit_b2', 'MIT-B3': 'mit_b3'}
ENCODER = _ZOO[MODEL]
IMG_H, IMG_W = IMG_SIZE

print(f'MODEL: DeepLabV3+ ({ENCODER}) | {IMG_H}x{IMG_W} | BS={BATCH_SIZE}')

MODEL: DeepLabV3+ (mit_b5) | 768x768 | BS=8


In [2]:
import os, sys, subprocess, shutil, glob
from pathlib import Path

print("[1/6] OFFLINE SETUP")

# === Tim offline_assets ===
ASSET_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "offline_assets" in dirs:
        ASSET_DIR = os.path.join(root, "offline_assets")
        break
    if "packages" in dirs and ("weights" in dirs or "dot_cache" in dirs):
        ASSET_DIR = root
        break

if not ASSET_DIR:
    raise RuntimeError("Khong tim thay offline_assets!")
print(f"  Found: {ASSET_DIR}")

# === Install smp ===
pkg_dir = os.path.join(ASSET_DIR, "packages")
if os.path.exists(pkg_dir) and glob.glob(os.path.join(pkg_dir, "*.whl")):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", pkg_dir,
        "segmentation-models-pytorch", "--quiet",
    ])
    print("  [OK] smp installed")

# === KHOI PHUC CACHE ===
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

home_cache = os.path.expanduser("~/.cache")
HF_HUB = os.path.join(home_cache, "huggingface", "hub")
TORCH_CKPT = os.path.join(home_cache, "torch", "hub", "checkpoints")
os.makedirs(HF_HUB, exist_ok=True)
os.makedirs(TORCH_CKPT, exist_ok=True)
copied = 0

# --- Cach 1: dot_cache/ (NB1 moi) ---
dot_cache = os.path.join(ASSET_DIR, "dot_cache")
if os.path.exists(dot_cache):
    print("  Using dot_cache/ (new format)")
    for root, dirs, files in os.walk(dot_cache):
        for f in files:
            src = os.path.join(root, f)
            rel = os.path.relpath(src, dot_cache)
            dst = os.path.join(home_cache, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst):
                try: os.symlink(src, dst)
                except OSError: shutil.copy2(src, dst)
                copied += 1

# --- Cach 2: weights/hub/ (NB1 cu) ---
else:
    weights_hub = os.path.join(ASSET_DIR, "weights", "hub")
    if os.path.exists(weights_hub):
        print("  Using weights/hub/ (old format)")
        for item in os.listdir(weights_hub):
            src = os.path.join(weights_hub, item)
            if item.startswith("models--"):
                dst = os.path.join(HF_HUB, item)
                if not os.path.exists(dst):
                    try: os.symlink(src, dst)
                    except OSError: shutil.copytree(src, dst)
                    copied += 1
                    print(f"    -> {item}")
            elif item == "checkpoints":
                for ckpt in os.listdir(src):
                    s = os.path.join(src, ckpt)
                    d = os.path.join(TORCH_CKPT, ckpt)
                    if not os.path.exists(d):
                        try: os.symlink(s, d)
                        except OSError: shutil.copy2(s, d)
                        copied += 1
                        print(f"    -> checkpoints/{ckpt}")
    else:
        print("  [WARN] No weights found!")

print(f"  [OK] Restored {copied} items")

# Verify
if os.path.exists(HF_HUB):
    models = [d for d in os.listdir(HF_HUB) if d.startswith("models--")]
    print(f"  HF models: {models}")
if os.path.exists(TORCH_CKPT):
    files = os.listdir(TORCH_CKPT)
    if files: print(f"  Torch checkpoints: {files}")
print("  [OK] Done")

[1/6] OFFLINE SETUP
  Found: /kaggle/input/datasets/hinlphmhng/uav-seg-offline-assets/offline_assets
  [OK] smp installed
  Using dot_cache/ (new format)
  [OK] Restored 37 items
  HF models: ['models--smp-hub--mit_b5.imagenet', 'models--smp-hub--resnet50.imagenet', 'models--smp-hub--mit_b3.imagenet', 'models--timm--convnext_large.fb_in22k_ft_in1k', 'models--smp-hub--efficientnet-b5.imagenet', 'models--smp-hub--resnet34.imagenet', 'models--timm--swinv2_large_window12to24_192to384.ms_in22k_ft_in1k', 'models--timm--convnext_base.fb_in22k_ft_in1k', 'models--smp-hub--resnet101.imagenet', 'models--timm--swinv2_base_window12to24_192to384.ms_in22k_ft_in1k']
  [OK] Done


In [3]:
import re, time, json, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

import segmentation_models_pytorch as smp
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/outputs')
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()} ({torch.cuda.get_device_properties(0).total_memory/1024**3:.0f} GB)')
else:
    print('[WARN] No GPU!')

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95 GB)


/usr/local/lib/python3.12/dist-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


In [11]:
_gdrive = None
_folder_id = None
try:
    from kaggle_secrets import UserSecretsClient
    from google.oauth2.service_account import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    s = UserSecretsClient()
    _gdrive = build('drive', 'v3', credentials=Credentials.from_service_account_info(
        json.loads(s.get_secret('GDRIVE_CREDENTIALS')),
        scopes=['https://www.googleapis.com/auth/drive.file']))
    _folder_id = s.get_secret('GDRIVE_FOLDER_ID')
    print('[OK] Google Drive enabled')
except:
    print('[SKIP] Google Drive')

def upload_to_drive(filepath):
    if not _gdrive: return
    try:
        f = Path(filepath)
        old = _gdrive.files().list(q=f"name='{f.name}' and '{_folder_id}' in parents and trashed=false", fields='files(id)').execute().get('files', [])
        media = MediaFileUpload(str(f), mimetype='application/octet-stream', resumable=True)
        if old: _gdrive.files().update(fileId=old[0]['id'], media_body=media).execute()
        else: _gdrive.files().create(body={'name': f.name, 'parents': [_folder_id]}, media_body=media).execute()
        print(f'    [DRIVE] {f.name}')
    except Exception as e:
        print(f'    [DRIVE ERR] {e}')

[SKIP] Google Drive


In [4]:
print('[2/6] DATA')

NUM_CLASSES = 11
LABEL_COLORS = [
    (0,255,255), (0,127,0), (19,132,69), (0,53,65), (130,76,0),
    (152,251,152), (151,126,171), (250,150,0), (115,176,195),
    (123,123,123), (0,0,0),
]
CLASS_NAMES = [
    'Sky', 'Deciduous_trees', 'Coniferous_trees', 'Fallen_trees',
    'Dirt_ground', 'Ground_vegetation', 'Rocks', 'Building',
    'Fence', 'Car', 'Empty',
]

def rgb_to_mask(rgb):
    out = np.full(rgb.shape[:2], NUM_CLASSES - 1, dtype=np.int64)
    for i, c in enumerate(LABEL_COLORS):
        out[np.all(rgb == c, axis=-1)] = i
    return out

DATA_ROOT = Path('/kaggle/working/data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

found = []
for dirpath, dirnames, filenames in os.walk('/kaggle/input', followlinks=True):
    if os.path.basename(dirpath) == 'color':
        seq_dir = os.path.dirname(dirpath)
        seq_name = os.path.basename(seq_dir)
        if re.match(r'seq\d+', seq_name):
            link = DATA_ROOT / seq_name
            if not link.exists(): os.symlink(seq_dir, link)
            found.append(seq_name)
            print(f'  {seq_name} -> {seq_dir}')

seqs = sorted(set(found))
print(f'Found {len(seqs)} sequences')
if not seqs: raise RuntimeError('NO DATA!')

[2/6] DATA
  seq6 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq6/seq6
  seq7 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq7/seq7
  seq8 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq8/seq8
  seq5 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq5/seq5
  seq2 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq2/seq2
  seq4 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq4/seq4
  seq3 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq3/seq3
  seq1 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq1/seq1
Found 8 sequences


In [5]:
class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples = []
        root = Path(root)
        for seq in sequences:
            cd = root / seq / 'color'
            if not cd.exists(): continue
            for p in sorted(cd.glob('*.png')):
                lbl = root / seq / 'labels' / p.name
                if lbl.exists(): self.samples.append((p, lbl))
        print(f'  {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img = np.array(Image.open(self.samples[i][0]).convert('RGB'))
        lbl = np.array(Image.open(self.samples[i][1]).convert('RGB'))
        mask = rgb_to_mask(lbl)
        if self.transform:
            r = self.transform(image=img, mask=mask)
            img, mask = r['image'], r['mask']
        return img.float(), mask.long()

train_seqs = [s for s in TRAIN_SEQS if s in seqs]
val_seqs = [s for s in VAL_SEQS if s in seqs]
if not train_seqs:
    train_seqs = seqs[:-1] if len(seqs) > 1 else seqs
    val_seqs = seqs[-1:] if len(seqs) > 1 else seqs
print(f'Train: {train_seqs} | Val: {val_seqs}')

train_tf = A.Compose([
    A.Resize(IMG_H, IMG_W), A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.1),
    A.Rotate(limit=15, p=0.3, border_mode=0),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3), A.HueSaturationValue(10, 20, 20, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_H, IMG_W),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), ToTensorV2(),
])

train_ds = ForestDataset(DATA_ROOT, train_seqs, train_tf)
val_ds = ForestDataset(DATA_ROOT, val_seqs, val_tf)
if len(train_ds) == 0: raise RuntimeError('Train EMPTY!')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {len(train_ds)} ({len(train_loader)} batches) | Val: {len(val_ds)}')

Train: ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8'] | Val: ['seq6']


/tmp/ipykernel_162/3965178634.py:35: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.2),


  10217 samples from ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq7', 'seq8']
  1437 samples from ['seq6']
Train: 10217 (1277 batches) | Val: 1437


In [6]:
print('[3/6] CLASS WEIGHTS')
n_sample = min(200, len(train_ds))
counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for idx in np.random.choice(len(train_ds), n_sample, replace=False):
    _, mask = train_ds[idx]
    for c in range(NUM_CLASSES): counts[c] += (mask == c).sum().item()
freq = counts / counts.sum()
for i, name in enumerate(CLASS_NAMES): print(f'  {name:20s} {freq[i]*100:6.2f}%')
w = np.ones(NUM_CLASSES, dtype=np.float32)
nz = freq > 0
w[nz] = 1.0 / (freq[nz] * NUM_CLASSES)
w = np.clip(w, 0.5, 10.0)
w = w / w.sum() * NUM_CLASSES
CLASS_WEIGHTS = torch.tensor(w, device=DEVICE)
print(f'Weights: {w.round(2).tolist()}')

[3/6] CLASS WEIGHTS
  Sky                   21.89%
  Deciduous_trees       13.79%
  Coniferous_trees      29.27%
  Fallen_trees           0.44%
  Dirt_ground            1.73%
  Ground_vegetation     30.86%
  Rocks                  1.79%
  Building               0.18%
  Fence                  0.02%
  Car                    0.04%
  Empty                  0.00%
Weights: [0.10000000149011612, 0.14000000059604645, 0.10000000149011612, 2.059999942779541, 1.0800000429153442, 0.10000000149011612, 1.0399999618530273, 2.059999942779541, 2.059999942779541, 2.059999942779541, 0.20999999344348907]


In [7]:
print('[4/6] BUILD MODEL')
model = smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES, activation=None)
print(f'  DeepLabV3+ ({ENCODER}): {sum(p.numel() for p in model.parameters()):,} params')
model = model.to(DEVICE)
print(f'  [OK] on {DEVICE}')

[4/6] BUILD MODEL
  DeepLabV3+ (mit_b5): 82,598,363 params
  [OK] on cuda


In [8]:
print('[5/6] LOSS + OPTIMIZER')

class FocalDiceLoss(nn.Module):
    def __init__(self, nc, cw=None, gamma=2.0, use_focal=True):
        super().__init__()
        self.nc, self.use_focal, self.gamma = nc, use_focal, gamma
        self.ce = nn.CrossEntropyLoss(weight=cw, reduction='none' if use_focal else 'mean')
    def forward(self, logits, targets):
        if self.use_focal:
            ce = self.ce(logits, targets)
            ce_term = ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()
        else: ce_term = self.ce(logits, targets)
        probs = F.softmax(logits, dim=1)
        tgt = F.one_hot(targets, self.nc).permute(0,3,1,2).float()
        inter = (probs * tgt).sum(dim=(0,2,3))
        card = probs.sum(dim=(0,2,3)) + tgt.sum(dim=(0,2,3))
        dice = 1.0 - ((2*inter + 1e-6) / (card + 1e-6)).mean()
        return ce_term + 0.5 * dice

criterion = FocalDiceLoss(NUM_CLASSES, CLASS_WEIGHTS, use_focal=FOCAL_LOSS)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
WARMUP = 5
scheduler = LambdaLR(optimizer, lambda e: e/WARMUP if e < WARMUP else 0.5*(1+math.cos(math.pi*(e-WARMUP)/max(1,EPOCHS-WARMUP))))
scaler = GradScaler('cuda', enabled=True)

start_epoch = 0
if RESUME and os.path.exists(RESUME):
    ckpt = torch.load(RESUME, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'  [RESUME] epoch {start_epoch}')
print('  [OK]')

[5/6] LOSS + OPTIMIZER
  [OK]


In [9]:
class SegMetrics:
    def __init__(self, n):
        self.n = n; self.cm = np.zeros((n,n), dtype=np.int64)
    def reset(self): self.cm[:] = 0
    @torch.no_grad()
    def update(self, preds, targets):
        p, t = preds.cpu().numpy().flatten(), targets.cpu().numpy().flatten()
        v = (t >= 0) & (t < self.n)
        self.cm += np.bincount(t[v]*self.n + p[v], minlength=self.n**2).reshape(self.n, self.n)
    def compute(self):
        inter = np.diag(self.cm)
        union = self.cm.sum(1) + self.cm.sum(0) - inter
        iou = np.where(union > 0, inter/union, 0.0)
        valid = union > 0
        return {'miou': float(iou[valid].mean()) if valid.any() else 0.0,
                'pixel_acc': float(inter.sum()/self.cm.sum()) if self.cm.sum()>0 else 0.0,
                'iou': iou}

In [ ]:
print('[6/6] TRAINING')
print('=' * 60)

best_miou = 0.0
no_improve = 0
saved_ckpts = []
metrics = SegMetrics(NUM_CLASSES)
history = {'train_loss': [], 'val_loss': [], 'miou': [], 'pixel_acc': [], 'lr': []}

for epoch in range(start_epoch + 1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tl, nt = 0.0, 0
    for imgs, masks in tqdm(train_loader, desc=f'E{epoch} train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast('cuda'): loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        tl += loss.item(); nt += 1
    train_loss = tl / nt

    model.eval()
    metrics.reset()
    vl, nv = 0.0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'E{epoch} val', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda'):
                logits = model(imgs)
                vl += criterion(logits, masks).item(); nv += 1
            metrics.update(logits.argmax(1), masks)
    val_loss = vl / nv
    r = metrics.compute()
    miou, pa = r['miou'], r['pixel_acc']

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['miou'].append(miou); history['pixel_acc'].append(pa); history['lr'].append(lr)

    print(f'E{epoch:3d}/{EPOCHS} | loss {train_loss:.4f}/{val_loss:.4f} | mIoU {miou:.4f} | pxAcc {pa:.4f} | lr {lr:.2e} | {time.time()-t0:.0f}s')

    if miou > best_miou:
        best_miou = miou; no_improve = 0
        path = CKPT_DIR / f'{MODEL}_e{epoch:03d}_miou{miou:.4f}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'scaler_state_dict': scaler.state_dict(), 'miou': miou,
                    'config': {'model': MODEL, 'encoder': ENCODER, 'img_size': IMG_SIZE}}, path)
        saved_ckpts.append((miou, path))
        saved_ckpts.sort(key=lambda x: x[0])
        while len(saved_ckpts) > 3:
            _, old = saved_ckpts.pop(0)
            if old.exists(): old.unlink()
        upload_to_drive(str(path))
        print(f'  >>> BEST mIoU: {miou:.4f}')
        for i, name in enumerate(CLASS_NAMES):
            if r['iou'][i] > 0: print(f'      {name:20s}: {r["iou"][i]:.4f}')
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f'Early stopping (patience={PATIENCE})'); break

print(f'\nDone! Best mIoU: {best_miou:.4f}')

[6/6] TRAINING


/tmp/ipykernel_162/3245362705.py:13: RuntimeWarning: invalid value encountered in divide
  iou = np.where(union > 0, inter/union, 0.0)


E  1/100 | loss 0.3059/0.3439 | mIoU 0.5832 | pxAcc 0.8539 | lr 1.20e-04 | 646s
  >>> BEST mIoU: 0.5832
      Deciduous_trees     : 0.6776
      Coniferous_trees    : 0.7371
      Fallen_trees        : 0.3757
      Dirt_ground         : 0.7665
      Ground_vegetation   : 0.7932
      Rocks               : 0.7127
      Building            : 0.8710
      Fence               : 0.2472
      Car                 : 0.6512


E  2/100 | loss 0.2102/0.3335 | mIoU 0.6410 | pxAcc 0.8694 | lr 1.80e-04 | 641s
  >>> BEST mIoU: 0.6410
      Sky                 : 0.0225
      Deciduous_trees     : 0.7031
      Coniferous_trees    : 0.7585
      Fallen_trees        : 0.4465
      Dirt_ground         : 0.8396
      Ground_vegetation   : 0.8111
      Rocks               : 0.7520
      Building            : 0.8762
      Fence               : 0.4592
      Car                 : 0.7414


E  3/100 | loss 0.1934/0.3288 | mIoU 0.6514 | pxAcc 0.8724 | lr 2.40e-04 | 641s
  >>> BEST mIoU: 0.6514
      Sky                 : 0.0391
      Deciduous_trees     : 0.7131
      Coniferous_trees    : 0.7669
      Fallen_trees        : 0.4340
      Dirt_ground         : 0.8435
      Ground_vegetation   : 0.8133
      Rocks               : 0.7435
      Building            : 0.9057
      Fence               : 0.4974
      Car                 : 0.7575


...

E 42/100 | loss 0.0986/0.2250 | mIoU 0.7046 | pxAcc 0.8919 | lr 1.96e-04 | 644s


E 43/100 | loss 0.0981/0.2244 | mIoU 0.7063 | pxAcc 0.8925 | lr 1.92e-04 | 645s


E 44/100 | loss 0.0967/0.2246 | mIoU 0.7054 | pxAcc 0.8927 | lr 1.87e-04 | 655s


E 45/100 | loss 0.0979/0.2234 | mIoU 0.7070 | pxAcc 0.8924 | lr 1.82e-04 | 649s


E46 train:  66%|██████▌   | 845/1277 [06:12<02:49,  2.55it/s]

In [ ]:
with open(OUTPUT_DIR / f'{MODEL}_history.json', 'w') as f: json.dump(history, f, indent=2)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 4, figsize=(22, 5))
ax[0].plot(history['train_loss'], label='Train'); ax[0].plot(history['val_loss'], label='Val')
ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(history['miou'], 'g'); ax[1].axhline(best_miou, color='r', ls='--', alpha=0.5)
ax[1].set_title(f'mIoU (best={best_miou:.4f})'); ax[1].grid(True, alpha=0.3)
ax[2].plot(history['pixel_acc'], 'b'); ax[2].set_title('Pixel Acc'); ax[2].grid(True, alpha=0.3)
ax[3].plot(history['lr'], 'orange'); ax[3].set_title('LR'); ax[3].grid(True, alpha=0.3)
fig.suptitle(f'DeepLabV3+ ({ENCODER}) @ {IMG_H}x{IMG_W}', fontweight='bold')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / f'{MODEL}_curves.png', dpi=150); plt.show()
if saved_ckpts: print(f'Best: {saved_ckpts[-1][1].name}')

---
# TEST SET EVALUATION (seq9)

**Evaluate model ngay sau khi train** — model da co trong memory, khong can load lai checkpoint.

Output: `test_results.json` + confusion matrix + qualitative visualization

In [ ]:
# ============================================================
# DOWNLOAD SEQ9 (Test Set) tu Zenodo
# ============================================================
import urllib.request, zipfile

SEQ9_DIR = Path('/kaggle/working/data/seq9') if IS_KAGGLE else Path('data/forest_sunny/seq9')
ZENODO_URL = 'https://zenodo.org/records/15511426/files/seq9.zip'

if SEQ9_DIR.exists() and (SEQ9_DIR / 'color').exists():
    n = len(list((SEQ9_DIR / 'color').glob('*.png')))
    print(f'[OK] seq9 da co: {n} images — skip download')
else:
    zip_path = Path('/kaggle/working/seq9.zip') if IS_KAGGLE else Path('data/seq9.zip')
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    SEQ9_DIR.parent.mkdir(parents=True, exist_ok=True)

    print(f'[DOWNLOAD] seq9 tu Zenodo...')
    def _progress(count, block_size, total_size):
        pct = count * block_size * 100 / total_size if total_size > 0 else 0
        mb  = count * block_size / 1024 / 1024
        print(f'\rProgress: {pct:.1f}% ({mb:.0f} MB)', end='', flush=True)

    urllib.request.urlretrieve(ZENODO_URL, str(zip_path), reporthook=_progress)
    print(f'\nDownloaded: {zip_path.stat().st_size / 1024**2:.0f} MB')

    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(SEQ9_DIR.parent)
    zip_path.unlink(missing_ok=True)

    n = len(list((SEQ9_DIR / 'color').glob('*.png')))
    print(f'[OK] seq9 ready: {n} images')

# Symlink cho Kaggle
if IS_KAGGLE:
    link = DATA_ROOT / 'seq9'
    if not link.exists():
        os.symlink(str(SEQ9_DIR), str(link))

assert (SEQ9_DIR / 'color').exists(), f'FAIL: {SEQ9_DIR}/color not found'
assert (SEQ9_DIR / 'labels').exists(), f'FAIL: {SEQ9_DIR}/labels not found'

In [ ]:
# ============================================================
# EVALUATE ON TEST SET (seq9)
# Model da co trong memory — KHONG can load lai checkpoint!
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

try:
    import seaborn as sns
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'seaborn', 'fvcore'])
    import seaborn as sns

try:
    from fvcore.nn import FlopCountAnalysis
except ImportError:
    FlopCountAnalysis = None

PALETTE = np.array(LABEL_COLORS, dtype=np.uint8)

def mask_to_rgb(mask):
    h, w = mask.shape
    out = np.zeros((h, w, 3), dtype=np.uint8)
    for i in range(NUM_CLASSES):
        out[mask == i] = PALETTE[i]
    return out

def calc_flops_eval(mdl):
    if FlopCountAnalysis is None: return -1.0
    try:
        dummy = torch.zeros(1, 3, *IMG_SIZE).to(DEVICE)
        fa = FlopCountAnalysis(mdl, dummy)
        fa.unsupported_ops_warnings(False)
        fa.uncalled_modules_warnings(False)
        return fa.total() / 1e9
    except: return -1.0

def calc_fps_eval(mdl, n_runs=50):
    if DEVICE != 'cuda': return -1.0
    mdl.eval()
    dummy = torch.zeros(1, 3, *IMG_SIZE).to(DEVICE)
    with torch.no_grad():
        for _ in range(10): mdl(dummy)
    torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_runs): mdl(dummy)
    torch.cuda.synchronize()
    return n_runs / (time.time() - t0)

# --- Load best checkpoint vao model hien tai ---
if saved_ckpts:
    best_path = saved_ckpts[-1][1]  # (miou, path), sorted ascending
    print(f'Loading best checkpoint: {best_path.name}')
    ckpt_data = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'])
    print(f'  Val mIoU: {ckpt_data.get("miou", "N/A")}')
else:
    print('[WARN] No saved checkpoints — using current model weights')

model.eval()

# --- Model info ---
params_M = sum(p.numel() for p in model.parameters()) / 1e6
flops_G  = calc_flops_eval(model)
fps      = calc_fps_eval(model)
print(f'Params: {params_M:.2f}M | FLOPs: {flops_G:.2f}G | FPS: {fps:.1f}')

# --- Test dataset ---
# Reuse ForestDataset class from training
test_ds = ForestDataset(DATA_ROOT, ['seq9'], val_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=(DEVICE=='cuda'))
print(f'Test set: {len(test_ds)} samples')

# --- Evaluate ---
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
saved_samples = []  # for qualitative viz

with torch.no_grad():
    for imgs, masks in tqdm(test_loader, desc='Testing'):
        imgs = imgs.to(DEVICE)
        with autocast('cuda', enabled=True):
            out = model(imgs)
        preds = out.argmax(1).cpu().numpy()
        gts   = masks.numpy()

        for pred, gt in zip(preds, gts):
            valid = (gt >= 0) & (gt < NUM_CLASSES)
            np.add.at(cm, (gt[valid], pred[valid]), 1)

        # Save first 8 samples
        if len(saved_samples) < 8:
            for i in range(min(len(imgs), 8 - len(saved_samples))):
                img_np = imgs[i].cpu().numpy().transpose(1,2,0)
                mean_n = np.array([0.485, 0.456, 0.406])
                std_n  = np.array([0.229, 0.224, 0.225])
                img_np = np.clip(img_np * std_n + mean_n, 0, 1)
                saved_samples.append((img_np, preds[i], gts[i]))

# --- Compute metrics ---
inter = np.diag(cm)
union = cm.sum(1) + cm.sum(0) - inter
iou   = np.where(union > 0, inter / union, np.nan)
test_miou = float(np.nanmean(iou))
test_pa   = float(inter.sum() / (cm.sum() + 1e-9))

prec = np.where(cm.sum(0) > 0, inter / cm.sum(0), np.nan)
rec  = np.where(cm.sum(1) > 0, inter / cm.sum(1), np.nan)
f1   = np.where((prec + rec) > 0, 2*prec*rec/(prec+rec), np.nan)
test_mf1 = float(np.nanmean(f1))

print(f'\n{"="*60}')
print(f'TEST RESULTS — DeepLabV3+ ({ENCODER})')
print(f'{"="*60}')
print(f'  mIoU:      {test_miou*100:.2f}%')
print(f'  Pixel Acc: {test_pa*100:.2f}%')
print(f'  mF1:       {test_mf1*100:.2f}%')
print(f'  Params:    {params_M:.2f}M')
print(f'  FLOPs:     {flops_G:.2f}G')
print(f'  FPS:       {fps:.1f}')
print(f'\nPer-class IoU:')
for i, name in enumerate(CLASS_NAMES):
    v = iou[i]
    s = f'{v*100:.2f}%' if not np.isnan(v) else 'N/A'
    print(f'  {name:20s}: {s}')

# --- Save results ---
EVAL_DIR = OUTPUT_DIR / 'eval_results'
EVAL_DIR.mkdir(parents=True, exist_ok=True)

test_results = {
    'model': 'deeplabv3plus', 'encoder': ENCODER, 'model_display': MODEL,
    'img_size': list(IMG_SIZE), 'best_val_miou': float(best_miou),
    'test_mIoU': round(test_miou * 100, 2),
    'test_PA':   round(test_pa * 100, 2),
    'test_mF1':  round(test_mf1 * 100, 2),
    'params_M':  round(params_M, 2),
    'flops_G':   round(flops_G, 2),
    'fps':       round(fps, 1),
    'per_class_iou': {CLASS_NAMES[i]: round(float(iou[i])*100, 2)
                      for i in range(NUM_CLASSES) if not np.isnan(iou[i])},
    'per_class_f1':  {CLASS_NAMES[i]: round(float(f1[i])*100, 2)
                      for i in range(NUM_CLASSES) if not np.isnan(f1[i])},
}

result_file = EVAL_DIR / f'deeplabv3plus_{ENCODER}_test_results.json'
with open(result_file, 'w') as f:
    json.dump(test_results, f, indent=2)
print(f'\nSaved: {result_file}')

# Save confusion matrix
np.save(EVAL_DIR / f'deeplabv3plus_{ENCODER}_cm.npy', cm)

# Save as CSV row (for easy aggregation in NB04)
import pandas as pd
row = {k: v for k, v in test_results.items() if k not in ('per_class_iou', 'per_class_f1', 'img_size')}
for i, cls in enumerate(CLASS_NAMES):
    v = iou[i]
    row[f'iou_{cls}'] = round(float(v)*100, 2) if not np.isnan(v) else -1.0
csv_path = EVAL_DIR / 'results_all.csv'
df_row = pd.DataFrame([row])
if csv_path.exists():
    df_old = pd.read_csv(csv_path)
    # Remove duplicate if exists
    mask = ~((df_old['model'] == row['model']) & (df_old['encoder'] == row['encoder']))
    df_old = df_old[mask]
    df_all = pd.concat([df_old, df_row], ignore_index=True)
else:
    df_all = df_row
df_all.to_csv(csv_path, index=False)
print(f'Saved: {csv_path} ({len(df_all)} rows total)')

In [ ]:
# ============================================================
# VIZ 1: Qualitative Predictions
# ============================================================
n = min(6, len(saved_samples))
fig, axes = plt.subplots(n, 3, figsize=(15, n * 4))
if n == 1: axes = axes[np.newaxis]

for i in range(n):
    img, pred, gt = saved_samples[i]
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Input', fontsize=11); axes[i, 0].axis('off')
    axes[i, 1].imshow(mask_to_rgb(gt))
    axes[i, 1].set_title('Ground Truth', fontsize=11); axes[i, 1].axis('off')
    axes[i, 2].imshow(mask_to_rgb(pred))
    axes[i, 2].set_title('Prediction', fontsize=11); axes[i, 2].axis('off')

patches = [mpatches.Patch(color=np.array(LABEL_COLORS[i])/255, label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.01),
           fontsize=9, frameon=True)
plt.suptitle(f'Qualitative Results — DeepLabV3+ ({ENCODER}) | Test mIoU={test_miou*100:.2f}%',
             fontsize=13, y=1.01)
plt.tight_layout()
out_path = EVAL_DIR / 'figure_qualitative.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ============================================================
# VIZ 2: Confusion Matrix
# ============================================================
cm_norm = cm.astype(float) / (cm.sum(1, keepdims=True) + 1e-9)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            ax=ax, linewidths=0.5, vmin=0, vmax=1)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_title(f'Normalized Confusion Matrix — DeepLabV3+ ({ENCODER})', fontsize=13)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
out_path = EVAL_DIR / 'figure_confusion_matrix.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

In [ ]:
# ============================================================
# SUMMARY
# ============================================================
print('=' * 60)
print('TRAIN + TEST COMPLETE!')
print('=' * 60)
print(f'  Model:     DeepLabV3+ ({ENCODER})')
print(f'  Val mIoU:  {best_miou:.4f}')
print(f'  Test mIoU: {test_miou*100:.2f}%')
print(f'  Test PA:   {test_pa*100:.2f}%')
print(f'  Test mF1:  {test_mf1*100:.2f}%')
print(f'\nOutputs:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size / 1024
        unit = 'KB' if sz < 1024 else 'MB'
        val = sz if sz < 1024 else sz / 1024
        print(f'  {f.relative_to(OUTPUT_DIR)} ({val:.1f} {unit})')
print(f'\nBUOC TIEP: Copy results_all.csv vao NB04 de tao paper figures + LaTeX tables')